In [0]:
from pyspark.sql.functions import (
    col,
    year,
    month,
    dayofmonth,
    date_format,
    round,
    when
)

# Read raw weather table
df = spark.table("portfolio.weather_api_pipeline.raw_weather_sao_paulo")

# Transform and enrich weather data
silver_df = (
    df
    .select(
        col("date"),
        col("city"),
        col("latitude"),
        col("longitude"),
        round(col("temperature_max"), 2).alias("temperature_max_celsius"),
        round(col("temperature_min"), 2).alias("temperature_min_celsius"),
        round(col("temperature_mean"), 2).alias("temperature_mean_celsius"),
        round(col("precipitation_sum"), 2).alias("precipitation_mm"),
        round(col("windspeed_max"), 2).alias("windspeed_max_kmh"),
        col("source"),
        col("ingestion_timestamp")
    )
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("day", dayofmonth(col("date")))
    .withColumn("month_name", date_format(col("date"), "MMMM"))
    .withColumn(
        "rain_flag",
        when(col("precipitation_mm") > 0, 1).otherwise(0)
    )
    .withColumn(
        "temperature_range_celsius",
        round(col("temperature_max_celsius") - col("temperature_min_celsius"), 2)
    )
    .orderBy("date")
)

# Save as silver Delta table
silver_df.write.mode("overwrite").saveAsTable(
    "portfolio.weather_api_pipeline.silver_weather_sao_paulo"
)

display(silver_df)